# Depression Detection — Exploratory Data Analysis

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

from data import load_twitter_dataset, preprocess
from features import linguistic_features

sns.set_theme(style='whitegrid')

## 1. Load & inspect

In [ ]:
df_raw = load_twitter_dataset()
print(df_raw.shape)
df_raw.head()

DatasetNotFoundError: Dataset 'mrm8488/depression-sampled' doesn't exist on the Hub or cannot be accessed.

In [ ]:
df = preprocess(df_raw)
df['label'].value_counts()

## 2. Class balance

In [ ]:
ax = df['label'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
ax.set_title('Class distribution')
ax.set_xlabel('Label')
ax.set_ylabel('Count')
plt.tight_layout()

## 3. Text length distribution

In [ ]:
df['text_len'] = df['text'].str.split().str.len()
fig, ax = plt.subplots()
for label, grp in df.groupby('label'):
    grp['text_len'].clip(upper=300).plot.hist(bins=50, alpha=0.6, label=label, ax=ax)
ax.set_title('Post length by class')
ax.set_xlabel('Word count')
ax.legend()
plt.tight_layout()

## 4. Word clouds

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, label in zip(axes, df['label'].unique()):
    corpus = ' '.join(df[df['label'] == label]['text'])
    wc = WordCloud(width=600, height=400, background_color='white').generate(corpus)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(label)
plt.tight_layout()

## 5. Linguistic features comparison

In [ ]:
import numpy as np
feat_names = ['log_length', 'first_person_rate', 'negation_rate', 'neg_emotion_rate', 'avg_word_len']
feats = linguistic_features(df['text'].tolist())
feat_df = pd.DataFrame(feats, columns=feat_names)
feat_df['label'] = df['label'].values
feat_df.groupby('label')[feat_names].mean().T.plot(kind='bar')
plt.title('Mean linguistic features by class')
plt.xticks(rotation=30)
plt.tight_layout()

## 6. Baseline Model — Train & Evaluate

In [ ]:
from scipy.sparse import hstack
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc
from features import build_tfidf, linguistic_features
from model import build_baseline
from data import split_dataset

train, val, test = split_dataset(df)

# Build TF-IDF on train only
tfidf = build_tfidf(train['text'].tolist())

def make_features(split_df):
    tfidf_mat = tfidf.transform(split_df['text'].tolist())
    ling_mat  = linguistic_features(split_df['text'].tolist())
    return hstack([tfidf_mat, ling_mat])

X_train = make_features(train)
X_val   = make_features(val)
X_test  = make_features(test)

baseline = build_baseline()
baseline.fit(X_train, train['label'])
print("Trained. Evaluating on validation set...")
val_preds = baseline.predict(X_val)
print(classification_report(val['label'], val_preds))

In [ ]:
# Confusion matrix on test set
test_preds = baseline.predict(X_test)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(test['label'], test_preds, ax=ax, colorbar=False)
ax.set_title('Baseline — Confusion Matrix (test)')
plt.tight_layout()

In [ ]:
# ROC curve on test set
label_binarized = (test['label'] == 'depression').astype(int)
proba = baseline.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(label_binarized, proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Baseline — ROC Curve (test)')
ax.legend()
plt.tight_layout()

In [ ]:
# Top predictive words (most positive/negative logistic regression weights)
import numpy as np

clf        = baseline.named_steps['clf']
vocab      = tfidf.get_feature_names_out()
n_ling     = 5  # linguistic features appended after tfidf
coefs      = clf.coef_[0][:len(vocab)]  # tfidf weights only

top_n = 20
top_pos_idx = np.argsort(coefs)[-top_n:][::-1]
top_neg_idx = np.argsort(coefs)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, indices, title, color in zip(
    axes,
    [top_pos_idx, top_neg_idx],
    ['Top words → depression', 'Top words → non-depression'],
    ['salmon', 'steelblue'],
):
    ax.barh([vocab[i] for i in indices][::-1], coefs[indices][::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Logistic regression weight')
plt.tight_layout()

## 7. Transformer Model — Fine-tune & Evaluate

> **Runtime note:** fine-tuning BERT takes ~10–20 min on CPU. Run on a GPU (Google Colab or a local GPU) for faster results. Set `USE_TRANSFORMER = True` below to run this section.

In [ ]:
USE_TRANSFORMER = True   # set to True to run fine-tuning
MODEL_NAME = "bert-base-uncased"  # or "mental/mental-bert-base-uncased"

In [ ]:
if USE_TRANSFORMER:
    import torch
    from datasets import Dataset
    from transformers import TrainingArguments, Trainer
    import evaluate as hf_evaluate
    from model import TransformerConfig, load_transformer

    config = TransformerConfig(model_name=MODEL_NAME)
    tokenizer, bert_model = load_transformer(config)

    label2id = config.label2id

    def tokenize(batch):
        return tokenizer(batch['text'], truncation=True,
                         max_length=config.max_length, padding='max_length')

    def encode_labels(batch):
        batch['labels'] = [label2id[l] for l in batch['label']]
        return batch

    def make_hf_dataset(split_df):
        return (
            Dataset.from_pandas(split_df[['text', 'label']])
            .map(encode_labels, batched=True)
            .map(tokenize, batched=True)
            .remove_columns(['text', 'label'])
        )

    hf_train = make_hf_dataset(train)
    hf_val   = make_hf_dataset(val)
    hf_test  = make_hf_dataset(test)

    f1_metric = hf_evaluate.load('f1')

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return f1_metric.compute(predictions=preds, references=labels, average='binary')

    training_args = TrainingArguments(
        output_dir='../checkpoints',
        num_train_epochs=config.epochs,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        learning_rate=config.learning_rate,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        report_to='none',
        logging_steps=50,
    )

    trainer = Trainer(
        model=bert_model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_val,
        compute_metrics=compute_metrics,
    )

    print(f"Fine-tuning {MODEL_NAME}...")
    trainer.train()
    print("Done.")

In [ ]:
if USE_TRANSFORMER:
    # Evaluate on test set
    test_results = trainer.evaluate(hf_test)
    print("Test results:", test_results)

    # Confusion matrix
    test_preds_raw = trainer.predict(hf_test)
    bert_preds = np.argmax(test_preds_raw.predictions, axis=-1)
    bert_labels = test_preds_raw.label_ids
    id2label = config.id2label

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        [id2label[l] for l in bert_labels],
        [id2label[p] for p in bert_preds],
        ax=ax, colorbar=False,
    )
    ax.set_title(f'{MODEL_NAME} — Confusion Matrix (test)')
    plt.tight_layout()

In [ ]:
if USE_TRANSFORMER:
    # ROC curve
    import torch.nn.functional as F
    import torch

    logits_tensor = torch.tensor(test_preds_raw.predictions)
    proba_bert = F.softmax(logits_tensor, dim=-1)[:, 1].numpy()

    fpr_b, tpr_b, _ = roc_curve(bert_labels, proba_bert)
    auc_b = auc(fpr_b, tpr_b)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr_b, tpr_b, label=f'{MODEL_NAME}  AUC = {auc_b:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', label='Random')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Transformer — ROC Curve (test)')
    ax.legend()
    plt.tight_layout()